In [1]:
print("Hello, World!")

Hello, World!


# 04_3 DQA-CWA v2 Scene-Client 12h Reproduction

This notebook gives the DQA-CWA runner the same shape as the FedSTO notebook: build the workspace, dry-run the pipeline, start or resume the experiment, inspect progress, and read the paper-style results in one place.

It assumes the shared BDD100K/FedSTO setup that already lives under `navigating_data_heterogeneity`, but keeps all DQA-specific runs and stats under `dynamic_quality_aware_classwise_aggregation/`.

In [2]:
from __future__ import annotations

import json
import shlex
import shutil
import socket
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2.py",
        "navigating_data_heterogeneity/setup_fedsto_exact_reproduction.py",
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend(
            [
                base,
                base / "Object_Detection",
                base / "masters_research" / "Object_Detection",
            ]
        )
    for candidate in candidate_dirs:
        if all((candidate / marker).exists() for marker in required):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Object_Detection repository root.")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
NAV_ROOT = REPO_ROOT / "navigating_data_heterogeneity"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_paper_protocol.py"
NOTEBOOK_GENERATOR = DQA_ROOT / "generate_dqa_cwa_notebook.py"
WORK_ROOT = DQA_ROOT / "efficientteacher_dqa_ver2_scene_12h"
STATS_ROOT = DQA_ROOT / "stats_dqa_ver2_scene_12h"
RUNNER_LOG = DQA_ROOT / "dqa_ver2_scene_12h_runner.out"
PID_PATH = DQA_ROOT / "dqa_ver2_scene_12h_runner.pid"
TRAIN_LOG = DQA_ROOT / "dqa_ver2_scene_12h_train.log"
FEDSTO_WORK_ROOT = NAV_ROOT / "efficientteacher_fedsto"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("dqa_root:", DQA_ROOT)
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)

repo_root: /app/Object_Detection
dqa_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation
workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h
stats_root: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa_ver2_scene_12h
python: /root/micromamba/envs/al_yolov8/bin/python


## 1. DQA v2 Scene-Client 12 Hour Configuration

This run keeps the DQA v2 aggregation implementation from 04, but changes the unlabeled clients from weather splits to BDD100K scene splits: highway, city street, and residential. The labeled server source remains cloudy, while the training-time validation list points at the union of the three scene validation splits so the run is easier to judge for the target environments. This notebook requires non-empty BDD100K det_v2 train/val JSON annotations because scene labels live only in the raw annotations.

In [3]:
WARMUP_EPOCHS = 20
PHASE1_ROUNDS = 20
PHASE2_ROUNDS = 35
DQA_START_PHASE = 2
BATCH_SIZE = 64
WORKERS = 0
REQUESTED_GPUS = 2
try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(
        f"Requested {REQUESTED_GPUS} GPUs, but {AVAILABLE_CUDA_GPUS} CUDA device(s) are visible. "
        f"Using GPUS={GPUS} to avoid DDP launch failure."
    )
def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(29515)
MIN_FREE_GIB = 70

APPEND_TRAIN_LOG = False
EXTRA_RUN_ARGS: list[str] = [
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]

{
    "warmup_epochs": WARMUP_EPOCHS,
    "phase1_rounds": PHASE1_ROUNDS,
    "phase2_rounds": PHASE2_ROUNDS,
    "dqa_start_phase": DQA_START_PHASE,
    "requested_gpus": REQUESTED_GPUS,
    "available_cuda_gpus": AVAILABLE_CUDA_GPUS,
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "gpus": GPUS,
    "master_port": MASTER_PORT,
    "min_free_gib": MIN_FREE_GIB,
    "workspace": str(WORK_ROOT),
    "stats_root": str(STATS_ROOT),
}

{'warmup_epochs': 20,
 'phase1_rounds': 20,
 'phase2_rounds': 35,
 'dqa_start_phase': 2,
 'requested_gpus': 2,
 'available_cuda_gpus': 2,
 'batch_size': 64,
 'workers': 0,
 'gpus': 2,
 'master_port': 29515,
 'min_free_gib': 70,
 'workspace': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h',
 'stats_root': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa_ver2_scene_12h'}

## Runtime Estimate

Using the completed 04 timing as a guide, 20 warm-up epochs, 20 phase-1 rounds, and 35 phase-2 rounds should land around 12 hours on the same 2x RTX 6000 Ada node. The exact time can move with scene-list size and DQA stats overhead.

## 2. Build the Shared Data Interface

This uses the same shared setup logic as FedSTO and refreshes the DQA workspace manifest, list files, and runtime configs.

In [4]:
subprocess.run(
    [
        str(PYTHON_BIN),
        str(RUN_SCRIPT),
        "--setup-only",
        "--workspace-root",
        str(WORK_ROOT),
        "--stats-root",
        str(STATS_ROOT),
    ],
    cwd=REPO_ROOT,
    check=True,
)

manifest_path = WORK_ROOT / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_summary = {
    "classes": manifest["classes"],
    "server_train_images": manifest["server"]["train_images"],
    "server_val_images": manifest["server"]["val_images"],
    "clients": [
        {
            "id": client["id"],
            "weather": client["weather"],
            "scene": client.get("scene", client.get("weather")),
            "images": client["images"],
        }
        for client in manifest["clients"]
    ],
    "paper_schedule": manifest["paper_schedule"],
}
manifest_summary

{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "c13b7365d5151ffb509aa30b344edd7b10c63c92",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "c13b7365d5151ffb509aa30b344edd7b10c63c92",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregatio

{'classes': ['person',
  'rider',
  'car',
  'bus',
  'truck',
  'bike',
  'motor',
  'traffic light',
  'traffic sign',
  'train'],
 'server_train_images': 4881,
 'server_val_images': 9864,
 'clients': [{'id': 0,
   'weather': 'highway',
   'scene': 'highway',
   'images': 5000},
  {'id': 1, 'weather': 'citystreet', 'scene': 'city street', 'images': 5000},
  {'id': 2, 'weather': 'residential', 'scene': 'residential', 'images': 5000}],
 'paper_schedule': {'warmup_epochs': 50,
  'phase1_rounds': 100,
  'phase2_rounds': 150,
  'local_epochs': 1}}

## 3. Dependency and Dry-Run Check

This makes sure the runtime can import the EfficientTeacher stack and that the DQA runner can generate commands without starting a real train job.

In [5]:
import importlib.util

required = {
    "yaml": "PyYAML",
    "cv2": "opencv-python",
    "thop": "thop",
    "tensorboard": "tensorboard",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Missing packages:", missing)
    raise ModuleNotFoundError("Missing runtime dependencies: " + ", ".join(missing))

subprocess.run(
    [
        str(PYTHON_BIN),
        str(RUN_SCRIPT),
        "--dry-run",
        "--workspace-root",
        str(WORK_ROOT),
        "--stats-root",
        str(STATS_ROOT),
        "--warmup-epochs",
        str(WARMUP_EPOCHS),
        "--phase1-rounds",
        str(PHASE1_ROUNDS),
        "--phase2-rounds",
        str(PHASE2_ROUNDS),
        "--dqa-start-phase",
        str(DQA_START_PHASE),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT),
        "--min-free-gib",
        str(MIN_FREE_GIB),
        *EXTRA_RUN_ARGS,
    ],
    cwd=REPO_ROOT,
    check=True,
)

{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "variant": "scene clients: highway, city street, residential",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "c13b7365d5151ffb509aa30b344edd7b10c63c92",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "c13b7365d5151ffb509aa30b344edd7b10c63c92",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/data_lists/paper_eval_scene_total_val.txt",
      "source_val_list": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregatio

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2.py', '--dry-run', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h', '--stats-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa_ver2_scene_12h', '--warmup-epochs', '20', '--phase1-rounds', '20', '--phase2-rounds', '35', '--dqa-start-phase', '2', '--batch-size', '64', '--workers', '0', '--gpus', '2', '--master-port', '29515', '--min-free-gib', '70', '--classwise-blend', '0.35', '--server-anchor', '1.25', '--localize-bn', '--enable-dqa-guard', '--dqa-drop-ratio-threshold', '0.15', '--dqa-spike-ratio-threshold', '3.0'], returncode=0)

## 4. Full DQA Reproduction Run

This notebook is meant to work cleanly with `Run All`: the cell below runs true DQA in the foreground with compact status updates, waits for completion, and leaves the later cells free to inspect metrics and evaluation outputs.

In [6]:
import os
import re
import signal
import time
from collections import deque

RUN_FULL_REPRODUCTION = True
ALLOW_CPU_TRAINING = False
FORCE_RESTART = False
FORCE_WARMUP = False
FORCE_RETRAIN = False
STATUS_EVERY_SECONDS = 60


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return None
    try:
        return int(text)
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(
        ["ps", "-o", "stat=", "-p", str(pid)],
        capture_output=True,
        text=True,
    )
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


cmd = [
    str(PYTHON_BIN),
    "-u",
    str(RUN_SCRIPT),
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--log-file",
    str(TRAIN_LOG),
]
if APPEND_TRAIN_LOG:
    cmd.append("--append-train-log")
if FORCE_RESTART:
    cmd.append("--force-restart")
if FORCE_WARMUP:
    cmd.append("--force-warmup")
if FORCE_RETRAIN:
    cmd.append("--force-retrain")
cmd.extend(EXTRA_RUN_ARGS)

IMPORTANT_OUTPUT = re.compile(
    r"(Resuming DQA-CWA after|No completed DQA-CWA federated rounds|Current global checkpoint|"
    r"Reusing completed (warm-up|DQA-CWA client run|DQA-CWA server run|DQA-CWA global checkpoint)|"
    r"Recovered DQA-CWA phase|Completed DQA-CWA phase|All requested DQA-CWA federated rounds|"
    r"Skipping DQA-CWA for phase|Falling back to BN-local FedAvg|"
    r"Dry run complete|Training failed|Traceback|RuntimeError|Exception|Error|out of memory|CUDA error)",
    re.IGNORECASE,
)


def compact_line(line: str, limit: int = 240) -> str:
    text = line.replace("\r", "").strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def tail_lines(path: Path, lines: int = 80) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(
            ["tail", "-n", str(lines), str(path)],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.splitlines()
    except (FileNotFoundError, subprocess.CalledProcessError):
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


def print_tail(path: Path, label: str, lines: int = 80) -> None:
    rows = tail_lines(path, lines)
    if not rows:
        print(f"{label}: not found or empty: {path}")
        return
    print(f"{label}: {path}")
    for row in rows:
        print(compact_line(row, limit=500))


if RUN_FULL_REPRODUCTION and AVAILABLE_CUDA_GPUS < 1 and not ALLOW_CPU_TRAINING:
    print(
        "No CUDA GPU is visible, so the full DQA run was not started. "
        "Use a GPU runtime, or set ALLOW_CPU_TRAINING = True if this is only a tiny debug run."
    )
elif RUN_FULL_REPRODUCTION:
    existing_pid = read_pid(PID_PATH)
    existing_state = pid_state(existing_pid)
    if existing_state not in {"missing", "zombie"}:
        raise RuntimeError(
            f"DQA runner is already active with PID {existing_pid} (state={existing_state}). "
            "Stop it or reuse that run before launching another foreground run."
        )

    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(cmd))
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)

    recent = deque(maxlen=20)
    last_status = time.monotonic()
    log_mode = "a" if APPEND_TRAIN_LOG else "w"
    with RUNNER_LOG.open(log_mode, encoding="utf-8", buffering=1) as runner_log:
        runner_log.write("\n\n===== DQA-CWA notebook run started =====\n")
        runner_log.write("Running: " + " ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            start_new_session=True,
        )
        PID_PATH.write_text(str(process.pid) + "\n", encoding="utf-8")
        assert process.stdout is not None
        try:
            for line in process.stdout:
                runner_log.write(line)
                recent.append(line)
                if IMPORTANT_OUTPUT.search(line):
                    print(compact_line(line))
                    last_status = time.monotonic()
                elif time.monotonic() - last_status >= STATUS_EVERY_SECONDS:
                    print(f"[running] full DQA log is still updating: {RUNNER_LOG}")
                    last_status = time.monotonic()
            return_code = process.wait()
        except KeyboardInterrupt:
            print("KeyboardInterrupt received; stopping the DQA runner and child training process.")
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    process.kill()
                process.wait(timeout=30)
            raise
        finally:
            if PID_PATH.exists():
                PID_PATH.unlink()
    if return_code != 0:
        print("Last captured lines:")
        for line in recent:
            text = compact_line(line)
            if text:
                print(text)
        print_tail(TRAIN_LOG, "Train log tail", lines=120)
        raise RuntimeError(
            f"DQA runner exited with status {return_code}. "
            f"See {RUNNER_LOG} and {TRAIN_LOG}."
        )
    print("DQA run completed successfully.")
else:
    print("Set RUN_FULL_REPRODUCTION = True and rerun this cell.")

Running: /root/micromamba/envs/al_yolov8/bin/python -u /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa_ver2_scene_12h --warmup-epochs 20 --phase1-rounds 20 --phase2-rounds 35 --dqa-start-phase 2 --batch-size 64 --workers 0 --gpus 2 --master-port 29515 --min-free-gib 70 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa_ver2_scene_12h_train.log --classwise-blend 0.35 --server-anchor 1.25 --localize-bn --enable-dqa-guard --dqa-drop-ratio-threshold 0.15 --dqa-spike-ratio-threshold 3.0
Runner log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa_ver2_scene_12h_runner.out
Train log: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa_ver2_scene_12h_train.log
Reusing comp

## 5. Inspect Progress, Stats Coverage, and Logs

This is the main heartbeat cell. It reports PID state, federated-round progress, global checkpoints, per-client stats coverage, and short tails of both logs.

In [7]:
def tail_lines(path: Path, lines: int = 25) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(
            ["tail", "-n", str(lines), str(path)],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.splitlines()
    except (FileNotFoundError, subprocess.CalledProcessError):
        from collections import deque

        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


history_path = WORK_ROOT / "history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
else:
    history = []

pid = read_pid(PID_PATH)
state = pid_state(pid)
completed_phase1 = sum(1 for entry in history if int(entry.get("phase", 0)) == 1)
completed_phase2 = sum(1 for entry in history if int(entry.get("phase", 0)) == 2)
expected_total = PHASE1_ROUNDS + PHASE2_ROUNDS
latest_global = Path(history[-1]["global"]) if history else WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"

free_gib = shutil.disk_usage(WORK_ROOT).free / 1024**3
stats_rows = []
client_count = len(manifest["clients"]) if "manifest" in globals() else 3
for phase, rounds in ((1, PHASE1_ROUNDS), (2, PHASE2_ROUNDS)):
    for round_idx in range(1, rounds + 1):
        round_file = STATS_ROOT / f"phase{phase}_round{round_idx:03d}.json"
        client_files = sorted(STATS_ROOT.glob(f"phase{phase}_round{round_idx:03d}_client*.json"))
        stats_rows.append(
            {
                "phase": phase,
                "round": round_idx,
                "round_stats": round_file.exists(),
                "client_stats": len(client_files),
                "expected_client_stats": client_count,
            }
        )

status_summary = {
    "pid": pid,
    "pid_state": state,
    "completed_phase1": f"{completed_phase1}/{PHASE1_ROUNDS}",
    "completed_phase2": f"{completed_phase2}/{PHASE2_ROUNDS}",
    "completed_total": f"{len(history)}/{expected_total}",
    "latest_global": str(latest_global),
    "free_gib": round(free_gib, 2),
}
display(pd.DataFrame([status_summary]))

if stats_rows:
    stats_df = pd.DataFrame(stats_rows)
    display(stats_df.tail(10))

print("Runner log tail:")
for line in tail_lines(RUNNER_LOG):
    print(line)

print("\nTrain log tail:")
for line in tail_lines(TRAIN_LOG):
    print(line)

,pid,pid_state,completed_phase1,completed_phase2,completed_total,latest_global,free_gib
0,None,missing,20/20,35/35,55/55,/app/Object_Detection/dynamic_quality_aware_cl...,89.87


,phase,round,round_stats,client_stats,expected_client_stats
45,2,26,True,3,3
46,2,27,True,3,3
47,2,28,True,3,3
48,2,29,True,3,3
49,2,30,True,3,3
50,2,31,True,3,3
51,2,32,True,3,3
52,2,33,True,3,3
53,2,34,True,3,3
54,2,35,True,3,3


Runner log tail:
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 29515 train.py --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/configs/runtime_phase2_client_round33_dqa_phase2_round033_client1_citystreet.yaml
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa_ver2_scene_12h_train.log
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 29515 train.py --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/configs/runtime_phase2_client_round33_dqa_phase2_round033_client2_residential.yaml
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/dqa_ver2_scene_12h_train.log
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 29515 train.py --cfg /a

## 6. Paper-Style Evaluation

This cell is intentionally quick by default: latest checkpoint, cloudy split, batch size 32, and plots disabled. Use the full split list only when you are ready for the slower paper table.

In [8]:
RUN_EVAL = False
EVAL_SPLITS = "cloudy"
EVAL_BATCH_SIZE = 32
EVAL_EXTRA_ARGS: list[str] = ["--no-plots"]

history_path = WORK_ROOT / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
eval_checkpoints = []
if history:
    latest = Path(history[-1]["global"])
    eval_checkpoints.append(f"latest_global={latest}")
elif (WORK_ROOT / "global_checkpoints" / "round000_warmup.pt").exists():
    eval_checkpoints.append(f"warmup_global={WORK_ROOT / 'global_checkpoints' / 'round000_warmup.pt'}")

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
]
for spec in eval_checkpoints:
    eval_cmd.extend(["--checkpoint", spec])
eval_cmd.extend(EVAL_EXTRA_ARGS)

if RUN_EVAL and eval_checkpoints:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
elif RUN_EVAL:
    print("Skipping evaluation because no global checkpoints exist yet:", WORK_ROOT / "global_checkpoints")

summary_path = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
if summary_path.exists():
    eval_summary = pd.read_csv(summary_path)
    display(eval_summary)
else:
    print("No DQA paper-protocol summary yet:", summary_path)
    print("Default evaluation is a quick cloudy/latest smoke test with plots disabled.")
    print("For the full table, set EVAL_SPLITS = 'cloudy,overcast,rainy,snowy,total' and add more checkpoints.")

No DQA paper-protocol summary yet: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/validation_reports/paper_protocol_eval_summary.csv
Default evaluation is a quick cloudy/latest smoke test with plots disabled.
For the full table, set EVAL_SPLITS = 'cloudy,overcast,rainy,snowy,total' and add more checkpoints.


## 7. Compare DQA and FedSTO Results

If both workspaces have paper-protocol summaries, this cell lines them up side by side.

In [9]:
comparison_paths = {
    "DQA-CWA": WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv",
    "FedSTO": FEDSTO_WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv",
}

frames = []
for method, path in comparison_paths.items():
    if path.exists():
        df = pd.read_csv(path)
        df.insert(0, "method", method)
        frames.append(df)

if not frames:
    print("No comparable paper-protocol summaries found yet.")
else:
    comparison = pd.concat(frames, ignore_index=True)
    display(comparison)

,method,checkpoint_label,checkpoint_path,split,split_list,status,returncode,images,labels,precision,recall,map50,map50_95,log_file,command,error
0,FedSTO,warmup,/app/Object_Detection/navigating_data_heteroge...,cloudy,/app/Object_Detection/navigating_data_heteroge...,ok,0,738.0,14937.0,0.784,0.4190,0.495,0.2790,/app/Object_Detection/navigating_data_heteroge...,/root/micromamba/envs/al_yolov8/bin/python val...,NaN
1,FedSTO,final,/app/Object_Detection/navigating_data_heteroge...,cloudy,/app/Object_Detection/navigating_data_heteroge...,ok,0,738.0,14937.0,0.162,0.0431,0.105,0.0639,/app/Object_Detection/navigating_data_heteroge...,/root/micromamba/envs/al_yolov8/bin/python val...,NaN


## 8. Artifact Index

Handy links for whatever we usually need next: manifest, logs, history, stats, checkpoints, and evaluation outputs.

In [10]:
def artifact_row(path: Path, label: str) -> dict:
    exists = path.exists()
    modified = (
        datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat()
        if exists
        else ""
    )
    return {
        "label": label,
        "path": str(path),
        "exists": exists,
        "modified_utc": modified,
    }


artifact_rows = [
    artifact_row(DQA_ROOT / "README.md", "readme"),
    artifact_row(NOTEBOOK_GENERATOR, "notebook_generator"),
    artifact_row(WORK_ROOT / "manifest.json", "manifest"),
    artifact_row(WORK_ROOT / "history.json", "history"),
    artifact_row(RUNNER_LOG, "runner_log"),
    artifact_row(TRAIN_LOG, "train_log"),
    artifact_row(WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv", "paper_eval_summary_csv"),
    artifact_row(WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.md", "paper_eval_summary_md"),
    artifact_row(WORK_ROOT / "global_checkpoints", "global_checkpoints_dir"),
    artifact_row(STATS_ROOT, "stats_dir"),
]
display(pd.DataFrame(artifact_rows))

,label,path,exists,modified_utc
0,readme,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-27T03:58:21.418132+00:00
1,notebook_generator,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T01:48:58.816786+00:00
2,manifest,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T11:03:55.478785+00:00
3,history,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T22:41:30.458457+00:00
4,runner_log,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T22:41:30.458457+00:00
5,train_log,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T22:41:29.718461+00:00
6,paper_eval_summary_csv,/app/Object_Detection/dynamic_quality_aware_cl...,False,
7,paper_eval_summary_md,/app/Object_Detection/dynamic_quality_aware_cl...,False,
8,global_checkpoints_dir,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T22:41:30.466457+00:00
9,stats_dir,/app/Object_Detection/dynamic_quality_aware_cl...,True,2026-04-28T22:38:48.077204+00:00
